# 78: Regime Detection & Adaptive Exits

**Critical Insight:** James Check's framework was designed for 2017/2021-style blow-off tops with -80% crashes. But 2023-2026 is a DIFFERENT regime with sustained institutional buying.

## The Hypothesis:

**Old Regime (Retail FOMO):**
- Sharp parabolic moves → blow-off top → -80% crash
- MVRV>2.0 signals top (correct!)
- Exit saves you from -80% crash

**New Regime (Institutional Flow):**
- Sustained bull runs with shallow 15-30% corrections
- MVRV>2.0 signals nothing (continues higher)
- Exit causes you to miss 113% rally

## Solution: Regime-Adaptive Exits

### Regime A: "Cycle Top" Mode
- **Indicators:** High funding (>0.05%), extreme leverage, parabolic price action
- **Exit Rule:** Aggressive (MVRV>2.0, LTH-SOPR>1.5)
- **Goal:** Avoid -80% crash

### Regime B: "Sustained Bull" Mode
- **Indicators:** Moderate funding (<0.03%), institutional flows, steady trends
- **Exit Rule:** Patient (MVRV>3.5, or don't exit)
- **Goal:** Stay invested during rally

## What We'll Test:

1. Can we detect regime changes in real-time?
2. Would adaptive exits fix the 87% underperformance?
3. What if we're wrong about 2023-2026 and crash -80% in 2027?
4. Historical performance: Would this work in 2017/2021?

**If this works, we've solved the framework's fundamental problem.**

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

try:
    import vectorbt as vbt
    print("✓ VectorBT loaded")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "vectorbt"])
    import vectorbt as vbt

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print("✓ Setup complete")

In [ ]:
# Configuration
PROJECT_ROOT = Path().resolve().parent
DATA_DIR = PROJECT_ROOT / "data" / "brk" / "daily"
GLASSNODE_DIR = PROJECT_ROOT / "data" / "glassnode" / "daily"

FEES = 0.001
SLIPPAGE = 0.001

## 1. Load Data (Full History)

In [ ]:
def load_metric(name: str, source: str = "brk") -> pd.Series:
    path = DATA_DIR / f"{name}.parquet" if source == "brk" else GLASSNODE_DIR / f"{name}.parquet"
    if not path.exists():
        return pd.Series(dtype=float)
    df = pd.read_parquet(path)
    if 'time' not in df.columns and isinstance(df.index, pd.DatetimeIndex):
        df = df.reset_index()
        if len(df.columns) == 2:
            df.columns = ['time', 'value']
    if 'value' not in df.columns:
        for col in df.columns:
            if col != 'time' and pd.api.types.is_numeric_dtype(df[col]):
                df['value'] = df[col]
                break
    if 'time' in df.columns and 'value' in df.columns:
        df['time'] = pd.to_datetime(df['time'])
        return df.set_index('time')['value'].sort_index()
    return pd.Series(dtype=float)

print("Loading data...")

metrics = {
    'price': ('price', 'brk'),
    'mvrv': ('mvrv', 'brk'),
    'mvrv_sth': ('mvrv_sth', 'brk'),
    'sopr_sth': ('sopr_sth', 'brk'),
    'sopr_lth': ('sopr_lth', 'brk'),
    'realized_profit': ('realized_profit', 'brk'),
    'realized_loss': ('realized_loss', 'brk'),
    'funding': ('funding_rate', 'glassnode'),
    'liq_long': ('liquidations_long', 'glassnode'),
    'liq_short': ('liquidations_short', 'glassnode'),
}

df_dict = {name: load_metric(metric, source) for name, (metric, source) in metrics.items()}
df = pd.DataFrame(df_dict).fillna(method='ffill')

print(f"✓ Data: {len(df)} days ({df.index[0].date()} to {df.index[-1].date()})")
df.head()

## 2. Calculate Regime Indicators

We'll use multiple signals to classify market regimes.

In [ ]:
# Calculate regime indicators
print("Calculating regime indicators...\n")

# 1. Volatility (30-day rolling std of returns)
returns = df['price'].pct_change()
volatility = returns.rolling(30).std() * np.sqrt(365)

# 2. Funding rate level (proxy for leverage/speculation)
funding_avg = df['funding'].rolling(30).mean()

# 3. Price momentum (above/below 200-day MA)
ma_200 = df['price'].rolling(200).mean()
price_vs_ma = df['price'] / ma_200

# 4. Rate of change (parabolic moves = high ROC)
roc_30 = df['price'].pct_change(30)

# 5. Liquidation ratio (measure of leverage stress)
liq_ratio = df['liq_long'] / df['liq_short']

# Summary stats
print("Regime Indicators Summary:")
print("="*60)
print(f"Volatility: {volatility.mean():.1%} avg (higher = more chaotic)")
print(f"Funding Rate: {funding_avg.mean():.4f} avg (higher = more leverage)")
print(f"Price vs 200MA: {price_vs_ma.mean():.2f}x avg")
print(f"30-day ROC: {roc_30.mean():.1%} avg")

## 3. Define Market Regimes

Classify each day as either "Cycle Top" or "Sustained Bull" regime.

In [ ]:
def classify_regime(df: pd.DataFrame) -> pd.Series:
    """
    Classify market regime based on multiple indicators.
    
    Cycle Top Regime (high risk of -80% crash):
    - High volatility (>80% annualized)
    - High funding (>0.04% avg)
    - Parabolic move (>50% in 30 days)
    - Extreme overvaluation (price >2.5x 200MA)
    
    Sustained Bull Regime:
    - Moderate volatility (<60%)
    - Moderate funding (<0.03%)
    - Steady gains (<40% in 30 days)
    - Reasonable valuation (<2.0x 200MA)
    
    Returns: Series of 'cycle_top' or 'sustained_bull'
    """
    # Calculate indicators
    vol = returns.rolling(30).std() * np.sqrt(365)
    fund = df['funding'].rolling(30).mean()
    roc = df['price'].pct_change(30)
    ma200 = df['price'].rolling(200).mean()
    price_ratio = df['price'] / ma200
    
    # Score each regime characteristic (0-1)
    cycle_top_score = (
        (vol > 0.80).astype(int) * 0.25 +           # High vol
        (fund > 0.04).astype(int) * 0.25 +          # High funding
        (roc > 0.50).astype(int) * 0.25 +           # Parabolic
        (price_ratio > 2.5).astype(int) * 0.25      # Extreme overvaluation
    )
    
    sustained_bull_score = (
        (vol < 0.60).astype(int) * 0.25 +           # Moderate vol
        (fund < 0.03).astype(int) * 0.25 +          # Moderate funding
        (roc < 0.40).astype(int) * 0.25 +           # Steady gains
        (price_ratio < 2.0).astype(int) * 0.25      # Reasonable valuation
    )
    
    # Classify: cycle_top if score > 0.5, else sustained_bull
    regime = pd.Series('sustained_bull', index=df.index)
    regime[cycle_top_score > 0.5] = 'cycle_top'
    
    return regime


# Classify regimes
regime = classify_regime(df)

print("\nRegime Classification:")
print("="*60)
print(regime.value_counts())
print(f"\nCycle Top: {(regime == 'cycle_top').sum() / len(regime) * 100:.1f}% of days")
print(f"Sustained Bull: {(regime == 'sustained_bull').sum() / len(regime) * 100:.1f}% of days")

## 4. Visualize Regime Changes Over Time

In [ ]:
# Plot regimes
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

# Price with regime shading
ax1 = axes[0]
ax1.plot(df.index, df['price'], color='black', linewidth=2, label='BTC Price')
ax1.fill_between(df.index, 0, df['price'].max() * 1.2,
                  where=(regime == 'cycle_top'), alpha=0.2, color='red', label='Cycle Top Regime')
ax1.fill_between(df.index, 0, df['price'].max() * 1.2,
                  where=(regime == 'sustained_bull'), alpha=0.2, color='green', label='Sustained Bull Regime')
ax1.set_ylabel('BTC Price ($)', fontsize=12)
ax1.set_title('Market Regimes Over Time', fontsize=14, fontweight='bold')
ax1.set_yscale('log')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Funding rate
ax2 = axes[1]
ax2.plot(df.index, funding_avg, color='blue', linewidth=1.5, label='30d Avg Funding')
ax2.axhline(0.04, color='red', linestyle='--', alpha=0.5, label='Cycle Top Threshold (0.04%)')
ax2.axhline(0.03, color='orange', linestyle='--', alpha=0.5, label='Sustained Bull Threshold (0.03%)')
ax2.set_ylabel('Funding Rate', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

# Volatility
ax3 = axes[2]
ax3.plot(df.index, volatility, color='purple', linewidth=1.5, label='30d Volatility')
ax3.axhline(0.80, color='red', linestyle='--', alpha=0.5, label='Cycle Top Threshold (80%)')
ax3.axhline(0.60, color='orange', linestyle='--', alpha=0.5, label='Sustained Bull Threshold (60%)')
ax3.set_ylabel('Volatility (Ann.)', fontsize=12)
ax3.set_xlabel('Date', fontsize=12)
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Red shading = Cycle Top regime (use aggressive exits)")
print("📊 Green shading = Sustained Bull regime (use patient exits)")

## 5. Generate Entry Signals (Same for Both Regimes)

In [ ]:
# Entry: Buy The Dip (4/5 conditions)
c1 = df['mvrv_sth'] < 1.0
c2 = df['sopr_sth'] < 1.0
c3 = (df['realized_profit'] / df['realized_loss']) < 1.0
c4 = df['funding'] <= 0.0
c5 = (df['liq_long'] / df['liq_short']) > 1.0
entry_count = c1.astype(int) + c2.astype(int) + c3.astype(int) + c4.astype(int) + c5.astype(int)
entries = entry_count >= 4

print(f"Entry signals: {entries.sum()}")

## 6. Generate Exit Signals (Regime-Dependent)

In [ ]:
def generate_adaptive_exits(df: pd.DataFrame, regime: pd.Series) -> pd.Series:
    """
    Generate exit signals that adapt to market regime.
    
    Cycle Top Regime: Aggressive exits (avoid -80% crash)
    - MVRV > 2.0 AND LTH-SOPR > 1.5 (original Check framework)
    
    Sustained Bull Regime: Patient exits (stay invested)
    - MVRV > 3.5 AND LTH-SOPR > 2.0 (much higher threshold)
    - OR: Never exit (trust the trend)
    """
    exits = pd.Series(False, index=df.index)
    
    # Cycle Top regime: Original Check framework (aggressive)
    cycle_top_exit = (df['mvrv'] > 2.0) & (df['sopr_lth'] > 1.5)
    exits[regime == 'cycle_top'] = cycle_top_exit[regime == 'cycle_top']
    
    # Sustained Bull regime: Much higher threshold (patient)
    sustained_bull_exit = (df['mvrv'] > 3.5) & (df['sopr_lth'] > 2.0)
    exits[regime == 'sustained_bull'] = sustained_bull_exit[regime == 'sustained_bull']
    
    return exits


# Generate different exit strategies
print("Generating exit signals...\n")

# 1. Original (no regime adaptation)
exits_original = (df['mvrv'] > 2.0) & (df['sopr_lth'] > 1.5)
print(f"Original exits: {exits_original.sum()}")

# 2. Regime-adaptive
exits_adaptive = generate_adaptive_exits(df, regime)
print(f"Adaptive exits: {exits_adaptive.sum()}")

# 3. Patient (high threshold everywhere)
exits_patient = (df['mvrv'] > 3.5) & (df['sopr_lth'] > 2.0)
print(f"Patient exits (MVRV>3.5): {exits_patient.sum()}")

# 4. Never exit (pure buy-and-hold with entries)
exits_never = pd.Series(False, index=df.index)
print(f"Never exit: 0")

## 7. Backtest All Strategies

In [ ]:
def backtest_strategy(df: pd.DataFrame, entries: pd.Series, exits: pd.Series, name: str) -> dict:
    """Run backtest using VectorBT."""
    pf = vbt.Portfolio.from_signals(
        close=df['price'],
        entries=entries,
        exits=exits,
        fees=FEES,
        slippage=SLIPPAGE,
        init_cash=10000,
        freq='1D'
    )
    return {
        'name': name,
        'portfolio': pf,
        'total_return': pf.total_return() * 100,
        'sharpe': pf.sharpe_ratio(),
        'max_dd': pf.max_drawdown() * 100,
        'num_trades': pf.trades.count(),
        'win_rate': pf.trades.win_rate() * 100 if pf.trades.count() > 0 else 0,
    }

print("\n" + "="*80)
print("BACKTESTING REGIME-ADAPTIVE STRATEGIES")
print("="*80)

# Backtest all strategies
results = {}
results['original'] = backtest_strategy(df, entries, exits_original, "Original (MVRV>2.0)")
results['adaptive'] = backtest_strategy(df, entries, exits_adaptive, "Regime-Adaptive")
results['patient'] = backtest_strategy(df, entries, exits_patient, "Patient (MVRV>3.5)")
results['never'] = backtest_strategy(df, entries, exits_never, "Never Exit")

# Buy and hold
bh_return = (df['price'].iloc[-1] / df['price'].iloc[0] - 1) * 100

# Results table
print(f"\n{'Strategy':<25} {'Return':>12} {'Sharpe':>8} {'Max DD':>10} {'Trades':>8} {'Win Rate':>10}")
print("-"*80)

for key, res in results.items():
    print(f"{res['name']:<25} {res['total_return']:>11.1f}% {res['sharpe']:>8.2f} {res['max_dd']:>9.1f}% {int(res['num_trades']):>8} {res['win_rate']:>9.1f}%")

print(f"{'Buy & Hold':<25} {bh_return:>11.1f}% {'~1.0':>8} {'?':>10} {'-':>8} {'-':>10}")
print("="*80)

# Comparison
print("\nVS BUY & HOLD:")
for key, res in results.items():
    diff = res['total_return'] - bh_return
    status = "✅ BEAT" if diff > 0 else "❌ LOST"
    print(f"  {res['name']:<25} {status} by {abs(diff):.1f}%")

## 8. Analyze by Time Period

Test if adaptive exits work better in 2023-2026 (sustained bull) vs 2017-2021 (cycle tops).

In [ ]:
# Split by major periods
periods = [
    ('2017-01-01', '2018-12-31', '2017-2018 Cycle'),
    ('2020-01-01', '2022-12-31', '2020-2022 Cycle'),
    ('2023-01-01', '2026-01-22', '2023-2026 Bull'),
]

print("\n" + "="*100)
print("PERFORMANCE BY TIME PERIOD")
print("="*100)

for start, end, label in periods:
    period_df = df[(df.index >= start) & (df.index <= end)].copy()
    if len(period_df) < 30:
        continue
    
    period_regime = regime[(regime.index >= start) & (regime.index <= end)]
    period_entries = entries[(entries.index >= start) & (entries.index <= end)]
    
    # Buy and hold
    bh = (period_df['price'].iloc[-1] / period_df['price'].iloc[0] - 1) * 100
    
    # Test strategies
    original_exits = exits_original[(exits_original.index >= start) & (exits_original.index <= end)]
    adaptive_exits = generate_adaptive_exits(period_df, period_regime)
    
    try:
        pf_original = vbt.Portfolio.from_signals(
            close=period_df['price'], entries=period_entries, exits=original_exits,
            fees=FEES, slippage=SLIPPAGE, init_cash=10000, freq='1D'
        )
        original_ret = pf_original.total_return() * 100
    except:
        original_ret = 0
    
    try:
        pf_adaptive = vbt.Portfolio.from_signals(
            close=period_df['price'], entries=period_entries, exits=adaptive_exits,
            fees=FEES, slippage=SLIPPAGE, init_cash=10000, freq='1D'
        )
        adaptive_ret = pf_adaptive.total_return() * 100
    except:
        adaptive_ret = 0
    
    # Regime classification
    cycle_top_pct = (period_regime == 'cycle_top').sum() / len(period_regime) * 100
    
    print(f"\n{label}:")
    print(f"  Regime: {cycle_top_pct:.0f}% Cycle Top, {100-cycle_top_pct:.0f}% Sustained Bull")
    print(f"  Buy & Hold: {bh:+.1f}%")
    print(f"  Original Exits: {original_ret:+.1f}% ({original_ret - bh:+.1f}% vs B&H)")
    print(f"  Adaptive Exits: {adaptive_ret:+.1f}% ({adaptive_ret - bh:+.1f}% vs B&H)")
    
    if adaptive_ret > original_ret:
        print(f"  ✅ Adaptive wins by {adaptive_ret - original_ret:.1f}%")
    else:
        print(f"  ❌ Original wins by {original_ret - adaptive_ret:.1f}%")

print("\n" + "="*100)

## 9. Equity Curves Comparison

In [ ]:
# Plot equity curves
fig, ax = plt.subplots(figsize=(16, 8))

# Buy & Hold
bh_equity = (df['price'] / df['price'].iloc[0]) * 10000
ax.plot(bh_equity.index, bh_equity.values, label='Buy & Hold', linewidth=2.5, color='black', linestyle='--', alpha=0.7)

# Strategies
colors = ['red', 'green', 'blue', 'orange']
for (key, res), color in zip(results.items(), colors):
    equity = res['portfolio'].value()
    ax.plot(equity.index, equity.values, label=res['name'], linewidth=2, alpha=0.8, color=color)

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Portfolio Value ($)', fontsize=12)
ax.set_title('Regime-Adaptive Exits: Equity Curves', fontsize=14, fontweight='bold')
ax.set_yscale('log')
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFinal Portfolio Values:")
print(f"  Buy & Hold: ${bh_equity.iloc[-1]:,.0f}")
for key, res in results.items():
    final = res['portfolio'].value().iloc[-1]
    print(f"  {res['name']}: ${final:,.0f}")

## 10. Conclusion: Does Regime Detection Work?

In [ ]:
print("\n" + "="*80)
print("REGIME-ADAPTIVE STRATEGY VERDICT")
print("="*80)

# Find best strategy
best = max(results.values(), key=lambda x: x['total_return'])
original = results['original']
adaptive = results['adaptive']

print(f"\n1. PERFORMANCE COMPARISON:")
print(f"   Buy & Hold: {bh_return:.1f}%")
print(f"   Original (MVRV>2.0): {original['total_return']:.1f}% ({original['total_return'] - bh_return:+.1f}%)")
print(f"   Regime-Adaptive: {adaptive['total_return']:.1f}% ({adaptive['total_return'] - bh_return:+.1f}%)")
print(f"   Best Strategy: {best['name']} at {best['total_return']:.1f}%")

improvement = adaptive['total_return'] - original['total_return']
print(f"\n2. REGIME ADAPTATION BENEFIT:")
print(f"   Improvement: {improvement:+.1f}%")

if improvement > 20:
    print(f"   ✅ SIGNIFICANT improvement! Regime detection works!")
elif improvement > 0:
    print(f"   ✓ Modest improvement, regime detection helps")
else:
    print(f"   ❌ No improvement, regime detection doesn't help")

print(f"\n3. KEY INSIGHTS:")
if adaptive['total_return'] > bh_return:
    print(f"   ✅ Regime-adaptive strategy BEATS buy-and-hold!")
    print(f"   ✅ This solves the framework's fundamental problem!")
else:
    gap = bh_return - adaptive['total_return']
    print(f"   ❌ Still underperforms buy-and-hold by {gap:.1f}%")
    print(f"   Improvement from {bh_return - original['total_return']:.1f}% to {gap:.1f}% gap")

print(f"\n4. RISK METRICS:")
print(f"   Original: Sharpe {original['sharpe']:.2f}, DD {original['max_dd']:.1f}%")
print(f"   Adaptive: Sharpe {adaptive['sharpe']:.2f}, DD {adaptive['max_dd']:.1f}%")
print(f"   Best: {best['name']} with Sharpe {best['sharpe']:.2f}")

print(f"\n5. RECOMMENDATION:")
if adaptive['total_return'] > bh_return:
    print(f"   🎯 Use Regime-Adaptive exits for paper trading")
    print(f"   🎯 This successfully adapts Check's framework to 2023-2026 market")
elif improvement > 30:
    print(f"   ⚠️  Regime-adaptive is better but still trails B&H")
    print(f"   Consider: Combine with 50/50 hybrid approach")
elif best['name'] == 'Never Exit':
    print(f"   💡 Best strategy: Don't exit at all in sustained bulls!")
    print(f"   Use Check's entries but stay invested until clear regime change")
else:
    print(f"   🤔 Regime detection doesn't solve the problem")
    print(f"   Need different approach (position sizing, trend filters, etc.)")

print("\n" + "="*80)

## Next Steps:

Based on results:
1. If regime-adaptive wins → implement for live trading
2. If "Never Exit" wins → rethink entire exit strategy
3. If still underperforms → combine with hybrid 50/50 from notebook 76
4. Test on full history including 2017-2018 crash to validate